# 03 – Embeddings & Clustering
Compute TF-IDF–weighted GloVe embeddings and optionally run agglomerative clustering.

**Inputs** : `outputs/interactions/pol_interactions.json`  
**Outputs**: Cluster labels added to the DataFrame.

In [ ]:
import logging, sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import gensim.downloader as api

from src.data.loaders import load_config
from src.topic_models.btm_model import preprocess
from src.embeddings.vectorizer import (
    TfidfEmbeddingVectorizer, cluster_documents,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
cfg = load_config('configs/config.yaml')
print('Config loaded.')

## 1. Load and preprocess interactions

In [ ]:
src_path = cfg['outputs']['interactions_dir'] + 'pol_interactions.json'
df = pd.read_json(src_path)
df = df[df['type'] != 'rt'].copy()

df['tokenized_text'] = df['full_text'].apply(preprocess)
df = df[df['tokenized_text'].apply(len) > 0].drop_duplicates('id')
print(f'Documents: {len(df)}')

## 2. Load GloVe embeddings

In [ ]:
emb_cfg = cfg['embeddings']
print(f"Loading {emb_cfg['model_name']} …")
model = api.load(emb_cfg['model_name'])
print('Embeddings loaded.')

## 3. Compute TF-IDF weighted document vectors

In [ ]:
texts = list(df['tokenized_text'])
vectorizer = TfidfEmbeddingVectorizer(model, dim=emb_cfg['dim'])
X = vectorizer.fit_transform(texts)
print(f'Embedding matrix shape: {X.shape}')

## 4. Agglomerative clustering

In [ ]:
cl_cfg = cfg['clustering']
labels = cluster_documents(
    X,
    n_clusters=cl_cfg['n_clusters'],
    linkage=cl_cfg['linkage'],
)
df['cluster'] = labels
print(df['cluster'].value_counts())

## 5. Inspect clusters

In [ ]:
for cluster_id in sorted(df['cluster'].unique()):
    print(f'\n--- Cluster {cluster_id} ({(labels==cluster_id).sum()} docs) ---')
    sample = df[df['cluster'] == cluster_id]['full_text'].head(3)
    for _, txt in sample.items():
        print(' •', txt[:120])